# Supervised Fine-Tuning (SFT) with Serverless customization on SageMaker AI

## Lab 3 - LLM Evaluation

In this notebook, we are going to run an Evaluation job on the fine-tuned model by using LLM as a Judge with Custom Metrics

***

In [ ]:
%load_ext autoreload
%autoreload 2

### Prerequisites

#### Setup and dependencies

> **Important — IAM setup for Bedrock evaluation:** 

The evaluation job uses Amazon Bedrock for LLM-as-Judge scoring. 

Your SageMaker execution role must include `bedrock.amazonaws.com` as a trusted entity in its trust policy. Without this, the pipeline will appear to start normally but the `EvaluateCustomModelMetrics` step will fail. 
You will see an error by inspecting the pipeline execution in the SageMaker console (Pipelines > Executions > select the failed step). 


See [Amazon Bedrock permissions setup](https://docs.aws.amazon.com/bedrock/latest/userguide/judge-service-roles.html) for instructions on updating your role's trust relationship.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Retrieve the fine-tuned model

In [ ]:
from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

In [ ]:
import hashlib

MAX_MPG_NAME_LENGTH = 63
suffix = "-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.core.resources import ModelPackageGroup

response = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

if len(response["ModelPackageSummaryList"]) > 0:
    fine_tuned_model_package_arn = response["ModelPackageSummaryList"][0]["ModelPackageArn"]
    fine_tuned_model_package_group_arn = ModelPackageGroup.get(model_package_group_name).model_package_group_arn
else:
    fine_tuned_model_package_arn = None
    fine_tuned_model_package_group_arn = None

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}/evaluation"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}/evaluation"

print(f"Model Package Group: {model_package_group_name}")
print(f"Model Package Group ARN: {fine_tuned_model_package_group_arn}")
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")
print(f"Evaluation output path: {output_path}")

test_dataset = DataSet.get(name="Multilingual-Thinking-sft-test")
print(f"Test dataset for evaluation: {test_dataset}")

***

### Create custom metrics for evaluation

The base model was fine-tuned on the [**Multilingual-Thinking**](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset (see [`1-prepare-data.ipynb`](1-prepare-data.ipynb)), where the model must **reason inside `<think>...</think>` tags in a target non-English language** and then give its **final answer in English**. The custom metrics below score exactly these behaviours, complementing the general-purpose built-in metrics:

- **ReasoningLanguageAdherence** — the reasoning inside `<think>` is written in the same non-English language as the reference reasoning.
- **EnglishFinalAnswer** — the final answer (after `</think>`) is written in fluent English.
- **ThinkTagStructure** — the response uses a single, well-formed `<think>...</think>` block followed by a non-empty answer.
- **ReasoningQuality** — the reasoning is on-topic, coherent, and logically supports the final answer.

> **Note:** LLM-as-Judge custom metrics only expose three placeholders — `{{prompt}}` (the input), `{{prediction}}` (the model's output) and `{{ground_truth}}` (the reference completion). The target reasoning language lives in the dataset's `system` field, which is **not** available to the judge, so `ReasoningLanguageAdherence` infers the expected language from the reference completion's `<think>` block instead. All custom metrics use a 0–1 scale to match the visualizations further down.

In [ ]:
import json

In [ ]:
EVALUATOR_MODEL = "amazon.nova-pro-v1:0"

In [ ]:
BUILTIN_METRICS = ["Correctness", "Completeness", "Faithfulness", "Coherence"]

# Custom metrics tailored to the Multilingual-Thinking task (see 1-prepare-data.ipynb):
# the model must reason inside <think>...</think> in a target non-English language,
# then give its final answer in English. Only {{prompt}}, {{prediction}} and
# {{ground_truth}} are available to the judge (there is no {{system}} placeholder),
# so the expected reasoning language is inferred from the reference completion.
# All metrics use a 0-1 scale to match the visualizations later in this notebook.
custom_metrics_list = [
    {
        "customMetricDefinition": {
            "name": "ReasoningLanguageAdherence",
            "instructions": (
                "The model was asked to write its reasoning (the text inside the "
                "<think>...</think> tags) in a specific non-English language. "
                "The reference response shows that target language. "
                "First, identify the language used inside the <think>...</think> tags "
                "of the reference. Then check whether the model's response writes its "
                "reasoning inside its own <think>...</think> tags in that SAME language. "
                "Judge the language only, not the correctness of the content.\n\n"
                "Prompt: {{prompt}}\n"
                "Response: {{prediction}}\n"
                "Reference: {{ground_truth}}"
            ),
            "ratingScale": [
                {
                    "definition": "Reasoning is in the same non-English target language as the reference",
                    "value": {"floatValue": 1},
                },
                {
                    "definition": "Reasoning is in a different language, or there is none to assess",
                    "value": {"floatValue": 0},
                },
            ],
        }
    },
    {
        "customMetricDefinition": {
            "name": "EnglishFinalAnswer",
            "instructions": (
                "Consider ONLY the final answer, i.e. the text that appears AFTER the "
                "closing </think> tag in the response. The model was instructed to give "
                "this final answer in English, regardless of the reasoning language. "
                "Judge the language of the final answer only, not its correctness.\n\n"
                "Prompt: {{prompt}}\n"
                "Response: {{prediction}}"
            ),
            "ratingScale": [
                {
                    "definition": "The final answer after </think> is in fluent, natural English",
                    "value": {"floatValue": 1},
                },
                {
                    "definition": "The final answer is in another language, empty, or missing",
                    "value": {"floatValue": 0},
                },
            ],
        }
    },
    {
        "customMetricDefinition": {
            "name": "ThinkTagStructure",
            "instructions": (
                "Check that the response follows the required format: exactly one "
                "well-formed <think>...</think> block containing the reasoning, "
                "immediately followed by a non-empty final answer OUTSIDE the tags. "
                "Penalize missing tags, unbalanced/duplicated tags, an empty <think> "
                "block, or a missing final answer.\n\n"
                "Response: {{prediction}}"
            ),
            "ratingScale": [
                {
                    "definition": "One well-formed <think>...</think> block, then a non-empty answer",
                    "value": {"floatValue": 1},
                },
                {
                    "definition": "Broken format: missing, empty, or unbalanced tags, or no answer",
                    "value": {"floatValue": 0},
                },
            ],
        }
    },
    {
        "customMetricDefinition": {
            "name": "ReasoningQuality",
            "instructions": (
                "Assess the quality of the reasoning inside the <think>...</think> tags, "
                "independently of the language it is written in. Good reasoning is "
                "on-topic for the question, internally coherent, and logically leads to "
                "the final answer given after </think>.\n\n"
                "Prompt: {{prompt}}\n"
                "Response: {{prediction}}\n"
                "Reference: {{ground_truth}}"
            ),
            "ratingScale": [
                {
                    "definition": "Excellent - coherent, on-topic reasoning that clearly supports the answer",
                    "value": {"floatValue": 1},
                },
                {
                    "definition": "Good - mostly sound reasoning with minor gaps",
                    "value": {"floatValue": 0.66},
                },
                {
                    "definition": "Poor - vague, partially off-topic, or weakly connected to the answer",
                    "value": {"floatValue": 0.33},
                },
                {
                    "definition": "Incorrect - reasoning is missing, irrelevant, or contradicts the answer",
                    "value": {"floatValue": 0},
                },
            ],
        }
    },
]

custom_metrics_json = json.dumps(custom_metrics_list)

In [ ]:
from sagemaker.train.evaluate import LLMAsJudgeEvaluator

evaluator = LLMAsJudgeEvaluator(
    model=fine_tuned_model_package_arn,
    model_package_group=fine_tuned_model_package_group_arn,
    evaluator_model=EVALUATOR_MODEL,  # Required
    dataset=test_dataset,  # Required: S3 URI or Dataset ARN
    builtin_metrics=BUILTIN_METRICS,  # Optional: Can combine with custom metrics
    custom_metrics=custom_metrics_json,  # Optional: JSON string of custom metrics
    s3_output_path=output_path,  # Required
    evaluate_base_model=False,  # Skip base model evaluation to evaluate only custom model
    sagemaker_session=sess,
)

In [ ]:
execution = evaluator.evaluate()

> **Tip:** The evaluation pipeline takes ~15-20 minutes. You can monitor progress in the SageMaker console under **Pipelines > Executions**. If the `EvaluateCustomModelMetrics` step fails with an "access denied" error, check that your execution role's trust policy includes `bedrock.amazonaws.com` (see the prerequisites note above).

In [ ]:
execution

***

### Analyze evaluation results

In this section we will further analyze the LLMAJ evaluation results produced by SageMaker AI serverless evaluation jobs, which are still accessible on S3.

In [ ]:
from rich.pretty import pprint
from sagemaker.train.common_utils import show_results_utils
from sagemaker.train.evaluate import EvaluationPipelineExecution
from sagemaker.train.evaluate.constants import EvalType

In [ ]:
# Use the evaluation we launched in this notebook rather than an arbitrary one.
# get_all() returns executions from every LLM-as-Judge pipeline in the account
# with no time ordering, so filtering only by "Succeeded" can pick an unrelated
# run. Prefer the `execution` object from evaluator.evaluate() above; fall back to
# matching this job's s3_output_path.
try:
    latest_succeeded = execution
except NameError:
    latest_succeeded = next(
        (
            e
            for e in EvaluationPipelineExecution.get_all(eval_type=EvalType.LLM_AS_JUDGE)
            if e.status.overall_status == "Succeeded"
            and getattr(e, "s3_output_path", None) == output_path
        ),
        None,
    )
pprint(latest_succeeded)

In [ ]:
_original_format = show_results_utils._format_score
show_results_utils._format_score = lambda s: (
    f"{s * 100:.1f}%" if s is not None else "N/A"
)

latest_succeeded.show_results(limit=5, offset=0, show_explanations=False)

show_results_utils._format_score = _original_format  # restore

#### Download results

First we download the results from S3 as JSONL files.

In [ ]:
import os
from urllib.parse import urlparse

In [ ]:
parsed = urlparse(latest_succeeded.s3_output_path)
bucket = parsed.netloc
prefix = parsed.path.lstrip("/")

In [ ]:
execution_id = latest_succeeded.arn.split("/")[-1]

paginator = s3_client.get_paginator("list_objects_v2")
candidates = []
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("_output.jsonl") and "custom-llmaj-eval" in key:
            # prefer files belonging to this run, but keep all as fallback
            priority = 1 if execution_id in key else 0
            candidates.append((priority, obj["LastModified"], key))

if not candidates:
    raise FileNotFoundError(
        f"No LLM-as-Judge results (custom-llmaj-eval .../*_output.jsonl) found under "
        f"s3://{bucket}/{prefix}. Confirm the evaluation job succeeded and wrote results."
    )

# Highest priority (this run), then most recent.
jsonl_key = max(candidates)[2]
print(f"Using results file: s3://{bucket}/{jsonl_key}")

os.makedirs("./tmp", exist_ok=True)

s3_client.download_file(bucket, jsonl_key, "./tmp/evaluation_results.jsonl")

#### Visualize results

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Utility functions used to create different charts

In [ ]:
def load_evaluation_results(filepath):
    """Load evaluation results from JSONL file into DataFrame."""
    with open(filepath) as f:
        results = [json.loads(line) for line in f]

    rows = []
    for r in results:
        for score in r["automatedEvaluationResult"]["scores"]:
            rows.append({"metric": score["metricName"], "score": score["result"]})

    return pd.DataFrame(rows)


def plot_metrics_bar(df):
    """Horizontal bar chart of average scores by metric."""
    agg = df.groupby("metric")["score"].mean().sort_values()

    plt.figure(figsize=(8, 5))
    bars = plt.barh(agg.index, agg.values, color="steelblue")
    plt.xlabel("Average Score")
    plt.title("LLM-as-Judge Evaluation Results")
    plt.xlim(0, 1)

    for bar, val in zip(bars, agg.values):
        plt.text(
            val + 0.02, bar.get_y() + bar.get_height() / 2, f"{val:.1%}", va="center"
        )

    plt.tight_layout()
    plt.show()


def plot_metrics_radar(df):
    """Radar chart showing all metrics."""
    agg = df.groupby("metric")["score"].mean()
    metrics = agg.index.tolist()
    values = agg.values.tolist() + [agg.values[0]]
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist() + [0]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, values, "o-", linewidth=2, color="steelblue")
    ax.fill(angles, values, alpha=0.25, color="steelblue")
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([m.replace("Builtin.", "") for m in metrics], size=9)
    ax.set_ylim(0, 1)
    ax.set_title("Evaluation Metrics Overview")
    plt.tight_layout()
    plt.show()


def plot_metrics_bullet(df, target=0.8):
    """Bullet chart comparing scores against target."""
    agg = df.groupby("metric")["score"].mean().sort_values()

    fig, ax = plt.subplots(figsize=(8, 4))
    y_pos = range(len(agg))
    ax.barh(y_pos, [1] * len(agg), color="#eee", height=0.6)
    ax.barh(y_pos, [target] * len(agg), color="#ddd", height=0.6)
    ax.barh(y_pos, agg.values, color="steelblue", height=0.3)
    ax.axvline(target, color="red", linestyle="--", label=f"Target ({target:.0%})")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(agg.index)
    ax.set_xlim(0, 1)
    ax.legend(loc="lower right")
    ax.set_title("Metrics vs Target")
    plt.tight_layout()
    plt.show()

In [ ]:
df = load_evaluation_results("./tmp/evaluation_results.jsonl")
plot_metrics_bar(df)
plot_metrics_radar(df)
plot_metrics_bullet(df, target=0.8)